In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

In [2]:
# Read all files in the data directory and load them as documents
def load_pdf(file_path):
    all_documents = []
    pdf_dir = Path(file_path)

    # Find all PDF files in the directory
    pdf_files = list(pdf_dir.glob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {file_path}")

    for pdf_file in pdf_files:
        print(f"Loading {pdf_file}...")
        loader = PyPDFLoader(str(pdf_file))
        documents = loader.load()
        for doc in documents:
            doc.metadata["source"] = str(pdf_file)
            doc.metadata["file_name"] = pdf_file.name
        all_documents.extend(documents)
        print(f"Loaded {len(documents)} documents from {pdf_file}")

    return all_documents

pdf_documents = load_pdf("../data")


Found 2 PDF files in ../data
Loading ..\data\attention.pdf...
Loaded 11 documents from ..\data\attention.pdf
Loading ..\data\rag.pdf...
Loaded 21 documents from ..\data\rag.pdf


In [3]:
pdf_documents

[Document(metadata={'source': '..\\data\\attention.pdf', 'page': 0, 'file_name': 'attention.pdf'}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser ∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence and convolutions\nentirely. Experiments on two machine translat

In [4]:
## Split the documents into smaller chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, separators=["\n\n", "\n", " ", ""], length_function=len)
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    return split_docs

In [5]:
chunks = split_documents(pdf_documents)
chunks

Split 32 documents into 187 chunks


[Document(metadata={'source': '..\\data\\attention.pdf', 'page': 0, 'file_name': 'attention.pdf'}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser ∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence and convolutions\nentirely. Experiments on two machine translat

### Embedding and Vetorstore DB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


c:\Users\Lenovo\Generative-ai\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        print(f"Loading embedding model: {self.model_name}...")
        self.model = SentenceTransformer(self.model_name)
        print("Model loaded successfully.")
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:  
        """Generate embeddings for a list of texts"""
        if self.model is None:
            raise ValueError("Model not loaded.")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print("Embeddings generated successfully.")
        return embeddings

In [8]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2...
Model loaded successfully.


### VectorStore

In [9]:
class VectorStore:
    """Manages the ChromaDB vector store for document embeddings"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()
    
    def _initialize_chromadb(self):
        """Initialize the ChromaDB client and collection"""
        print("Initializing ChromaDB client...")
        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        if self.collection_name in self.client.list_collections():
            print(f"Collection '{self.collection_name}' already exists. Loading existing collection.")
            self.collection = self.client.get_collection(name=self.collection_name)
        else:
            print(f"Creating new collection '{self.collection_name}'.")
            self.collection = self.client.create_collection(name=self.collection_name)
        print("ChromaDB client initialized successfully.")
    
    def add_embeddings(self, embeddings: np.ndarray, documents: List[Any]):
        """Add embeddings to the ChromaDB collection"""
        if len(embeddings) != len(documents):
            raise ValueError("Number of embeddings must match number of documents.")
        
        print(f"Adding {len(embeddings)} embeddings to the collection...")

        #Preparing data for Chroma DB
        ids, metadatas, documents_texts, embeddings_list = [], [], [], []

        for i , (doc, embeding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['context_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document conent
            documents_texts.append(doc.page_content)

            # Embeddings
            embeddings_list.append(embeding.tolist())
        
        #Add to collection
        try:

            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(embeddings)} embeddings to the collection.")
            print(f"Collection now contains {self.collection.count()} items.")
            print("Embeddings added successfully.")
        except Exception as e:
            print(f"Error adding embeddings to collection: {e}")

vector_store = VectorStore()
vector_store


Initializing ChromaDB client...
Creating new collection 'pdf_documents'.
ChromaDB client initialized successfully.


In [10]:
# Convert text to embeddings and add to vector store
def process_and_store_documents(documents):
    """Process documents to generate embeddings and store in ChromaDB"""
    print("Processing documents to generate embeddings...")
    texts = [doc.page_content for doc in documents]
    embeddings = embedding_manager.generate_embeddings(texts)
    vector_store.add_embeddings(embeddings, documents)
    print("Documents processed and stored successfully.")
    

In [11]:
process_and_store_documents(chunks)

Processing documents to generate embeddings...
Generating embeddings for 187 texts...


Batches: 100%|██████████| 6/6 [00:09<00:00,  1.60s/it]


Embeddings generated successfully.
Adding 187 embeddings to the collection...
Successfully added 187 embeddings to the collection.
Collection now contains 187 items.
Embeddings added successfully.
Documents processed and stored successfully.


### Retriever Pipeline from Vector Store

In [28]:

class Retriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, k: int = 5, score_threshold = 0) -> List[Dict[str, Any]]:
        """Retrieve relevant documents based on a query"""
        print(f"Retrieving documents for query: '{query}'")
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search inVector store
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=k
        )

        print(results)

        # Process results
        retrieved_docs = []
        if results['documents'] and results['documents'][0]:
            documents = results['documents'][0]
            metadatas = results['metadatas'][0]
            distance = results['distances'][0]
            ids = results['ids'][0]
            for i, (doc, meta, dist, doc_id) in enumerate(zip(documents, metadatas, distance, ids)):
                score = 1 - dist  # Convert distance to similarity score
                #print(score)
                retrieved_docs.append({
                         "id": doc_id,
                         "document": doc,
                         "metadata": meta,
                         "score": score,
                         "rank": i + 1
                     })
        #         if score >= score_threshold:
        #             retrieved_docs.append({
        #                 "id": doc_id,
        #                 "document": doc,
        #                 "metadata": meta,
        #                 "score": score,
        #                 "rank": i + 1
        #             })|
        #     print(f"Retrieved {len(retrieved_docs)} relevant documents.")
        # else:
        #     print("No relevant documents found.")
        return retrieved_docs
    


In [29]:
rag_retriever = Retriever(vector_store, embedding_manager)

In [20]:
rag_retriever

In [31]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 76.04it/s]

Embeddings generated successfully.
{'ids': [['doc_2eca6301_25', 'doc_bb850639_13', 'doc_86ac9a77_12', 'doc_0ccfaccc_21', 'doc_8e978bec_0']], 'embeddings': None, 'documents': [['convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,\nthe approach we take in our model.\nAs side beneﬁt, self-attention could yield more interpretable models. We inspect attention distributions\nfrom our models and present and discuss examples in the appendix. Not only do individual attention\nheads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic\nand semantic structure of the sentences.\n5 Training\nThis section describes the training regime for our models.\n5.1 Training Data and Batching\nWe trained on the standard WMT 2014 English-German dataset consisting of about 4.5 million\nsentence pairs. Sentences were encoded using byte-pair encoding [ 3], which has a shared source-\ntarget vocabulary of about 37000 to

[{'id': 'doc_2eca6301_25',
  'document': 'convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,\nthe approach we take in our model.\nAs side beneﬁt, self-attention could yield more interpretable models. We inspect attention distributions\nfrom our models and present and discuss examples in the appendix. Not only do individual attention\nheads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic\nand semantic structure of the sentences.\n5 Training\nThis section describes the training regime for our models.\n5.1 Training Data and Batching\nWe trained on the standard WMT 2014 English-German dataset consisting of about 4.5 million\nsentence pairs. Sentences were encoded using byte-pair encoding [ 3], which has a shared source-\ntarget vocabulary of about 37000 tokens. For English-French, we used the signiﬁcantly larger WMT\n2014 English-French dataset consisting of 36M sentences and split tokens